# Data Drift Monitoring with Evidently AI

**Module 2 - Notebook 6** | Skillsoft Track 3: ML Engineer

## Learning Objectives
- Detect data drift with Evidently AI
- Monitor feature distributions over time
- Set up automated drift reports with MLflow tracking
- Configure alerting thresholds and automated alerts
- Integrate with Airflow for scheduled monitoring

In [ ]:
# Setup
!pip install evidently==0.4.0 -q 2>/dev/null || pip install evidently -q
!pip install mlflow -q

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataDriftPreset, DataQualityPreset, TargetDriftPreset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import json
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow_drift.db")
experiment_name = "drift_monitoring"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_name)

print("Evidently AI + MLflow ready")



## 1. Generate Reference & Current Datasets

In [ ]:
# Reference dataset (training time)
np.random.seed(42)
X_ref, y_ref = make_classification(n_samples=5000, n_features=10, n_informative=8, random_state=42)
ref_df = pd.DataFrame(X_ref, columns=[f"feature_{i}" for i in range(10)])
ref_df['target'] = y_ref
ref_df['timestamp'] = datetime.now() - timedelta(days=30)

# Current dataset (simulated drift - feature_0 mean shifted)
np.random.seed(123)
X_curr, y_curr = make_classification(n_samples=5000, n_features=10, n_informative=8, random_state=123)
X_curr[:, 0] = X_curr[:, 0] + 2.0  # Introduce drift in feature_0
X_curr[:, 3] = X_curr[:, 3] * 1.5  # Scale drift in feature_3

curr_df = pd.DataFrame(X_curr, columns=[f"feature_{i}" for i in range(10)])
curr_df['target'] = y_curr
curr_df['timestamp'] = datetime.now()

print(f"Reference: {ref_df.shape}")
print(f"Current: {curr_df.shape}")

## 2. Data Drift Report with MLflow Tracking

In [ ]:
# Data drift report with MLflow logging
with mlflow.start_run(run_name="data_drift_check") as run:
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])

    data_drift_report.run(reference_data=ref_df.drop('target', axis=1), current_data=curr_df.drop('target', axis=1))

    # Save as HTML
    data_drift_report.save_html("data_drift_report.html")

    # Get results as dict
    drift_results = data_drift_report.as_dict()

    # Log to MLflow
    mlflow.log_param("reference_samples", len(ref_df))
    mlflow.log_param("current_samples", len(curr_df))
    
    # Extract and log drift metrics
    drift_summary = {"drifted_columns": [], "drift_scores": {}}
    for metric in drift_results['metrics']:
        if metric['metric'] == 'DatasetDriftMetric':
            drift_detected = metric['result']['dataset_drift']
            drift_share = metric['result']['drift_share']
            n_drifted = metric['result']['number_of_drifted_columns']
            n_total = metric['result']['number_of_columns']
            
            mlflow.log_metric("dataset_drift_detected", int(drift_detected))
            mlflow.log_metric("drift_share", drift_share)
            mlflow.log_metric("n_drifted_columns", n_drifted)
            mlflow.log_metric("total_columns", n_total)
            
            print(f"Dataset Drift Detected: {drift_detected}")
            print(f"Drift Share: {drift_share:.2%}")
            print(f"Drifted Columns: {n_drifted}/{n_total}")
        elif metric['metric'] == 'ColumnDriftMetric':
            col = metric['result']['column_name']
            drifted = metric['result']['drift_detected']
            score = metric['result']['drift_score']
            
            if drifted:
                print(f"  {col}: DRIFTED (score={score:.4f})")
                drift_summary["drifted_columns"].append(col)
                drift_summary["drift_scores"][col] = score
                mlflow.log_metric(f"drift_score_{col}", score)
    
    # Save drift summary as artifact
    import json
    with open("drift_summary.json", "w") as f:
        json.dump(drift_summary, f, indent=2)
    mlflow.log_artifact("drift_summary.json")
    
    # Save HTML report as artifact
    mlflow.log_artifact("data_drift_report.html")
    
    run_id = run.info.run_id

## 3. Data Quality Report

In [ ]:
# Data quality report
with mlflow.start_run(run_name="data_quality_check") as run:
    data_quality_report = Report(metrics=[
        DataQualityPreset(),
    ])

    data_quality_report.run(reference_data=ref_df, current_data=curr_df)
    data_quality_report.save_html("data_quality_report.html")
    mlflow.log_artifact("data_quality_report.html")
    
    print("Data quality report saved and logged to MLflow")



## 4. Target Drift Report (with predictions)

In [ ]:
# Train model on reference data
X_ref = ref_df.drop(['target', 'timestamp'], axis=1)
y_ref = ref_df['target']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_ref, y_ref)

# Add predictions to both datasets
ref_df['prediction'] = model.predict(X_ref)
ref_df['prediction_proba'] = model.predict_proba(X_ref)[:, 1]

X_curr = curr_df.drop(['target', 'timestamp'], axis=1)
curr_df['prediction'] = model.predict(X_curr)
curr_df['prediction_proba'] = model.predict_proba(X_curr)[:, 1]

# Target drift report
with mlflow.start_run(run_name="target_drift_check") as run:
    target_drift_report = Report(metrics=[
        TargetDriftPreset(),
    ])

    target_drift_report.run(reference_data=ref_df, current_data=curr_df)
    target_drift_report.save_html("target_drift_report.html")
    mlflow.log_artifact("target_drift_report.html")
    
    print("Target drift report saved and logged to MLflow")



## 5. Model Performance Drift (Prediction Drift)

In [ ]:
# Model performance drift - compare prediction distributions
with mlflow.start_run(run_name="prediction_drift_check") as run:
    from sklearn.metrics import accuracy_score, roc_auc_score
    
    # Reference performance
    X_ref = ref_df.drop(['target', 'timestamp', 'prediction', 'prediction_proba'], axis=1)
    y_ref = ref_df['target']
    ref_acc = accuracy_score(y_ref, ref_df['prediction'])
    ref_auc = roc_auc_score(y_ref, ref_df['prediction_proba'])
    
    # Current performance
    X_curr = curr_df.drop(['target', 'timestamp', 'prediction', 'prediction_proba'], axis=1)
    y_curr = curr_df['target']
    curr_acc = accuracy_score(y_curr, curr_df['prediction'])
    curr_auc = roc_auc_score(y_curr, curr_df['prediction_proba'])
    
    # Performance drift
    acc_drift = ref_acc - curr_acc
    auc_drift = ref_auc - curr_auc
    
    # Log to MLflow
    mlflow.log_metric("ref_accuracy", ref_acc)
    mlflow.log_metric("curr_accuracy", curr_acc)
    mlflow.log_metric("accuracy_drift", acc_drift)
    mlflow.log_metric("ref_auc", ref_auc)
    mlflow.log_metric("curr_auc", curr_auc)
    mlflow.log_metric("auc_drift", auc_drift)
    
    print(f"Reference: Acc={ref_acc:.4f}, AUC={ref_auc:.4f}")
    print(f"Current:   Acc={curr_acc:.4f}, AUC={curr_auc:.4f}")
    print(f"Drift:     Acc={acc_drift:.4f}, AUC={auc_drift:.4f}")
    
    # Alert if performance drops significantly
    if abs(acc_drift) > 0.05 or abs(auc_drift) > 0.05:
        print("⚠️ PERFORMANCE DRIFT DETECTED: Model performance degraded significantly")
        mlflow.log_param("performance_alert", "true")
    else:
        mlflow.log_param("performance_alert", "false")

## 6. Automated Monitoring Script with MLflow

In [ ]:
# Write monitoring script to file
monitoring_script = (
    "# Automated drift monitoring script with MLflow integration.\n"
    "\n"
    "import pandas as pd\n"
    "import numpy as np\n"
    "from datetime import datetime\n"
    "from evidently.legacy.report import Report\n"
    "from evidently.legacy.metric_preset import DataDriftPreset\n"
    "import json\n"
    "import os\n"
    "import mlflow\n"
    "from mlflow.tracking import MlflowClient\n"
    "import smtplib\n"
    "from email.mime.text import MIMEText\n"
    "\n"
    "DRIFT_THRESHOLD = 0.3  # Alert if drift share > 30%\n"
    "ACCURACY_DRIFT_THRESHOLD = 0.05  # Alert if accuracy drops > 5%\n"
    "ALERT_EMAIL = os.environ.get(\"ALERT_EMAIL\", \"\")\n"
    "MLFLOW_URI = os.environ.get(\"MLFLOW_URI\", \"sqlite:///mlflow.db\")\n"
    "\n"
    "mlflow.set_tracking_uri(MLFLOW_URI)\n"
    "client = MlflowClient()\n"
    "\n"
    "def check_drift(reference_path: str, current_path: str, model_path: str = None, output_dir: str = \"./reports\") -> dict:\n"
    "    \"\"\"Run drift check and return results with MLflow logging.\"\"\"\n"
    "    ref_df = pd.read_csv(reference_path)\n"
    "    curr_df = pd.read_csv(current_path)\n"
    "    \n"
    "    # Exclude non-feature columns\n"
    "    exclude_cols = [\"target\", \"timestamp\", \"prediction\", \"prediction_proba\"]\n"
    "    feature_cols = [c for c in ref_df.columns if c not in exclude_cols]\n"
    "    \n"
    "    # MLflow run\n"
    "    mlflow.set_tracking_uri(MLFLOW_URI)\n"
    "    mlflow.set_experiment(\"production_drift_monitoring\")\n"
    "    \n"
    "    with mlflow.start_run(run_name=f\"drift_check_{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}\") as run:\n"
    "        mlflow.log_param(\"reference_path\", reference_path)\n"
    "        mlflow.log_param(\"current_path\", current_path)\n"
    "        \n"
    "        # Data drift report\n"
    "        report = Report(metrics=[DataDriftPreset()])\n"
    "        report.run(reference_data=ref_df[feature_cols], current_data=curr_df[feature_cols])\n"
    "        \n"
    "        # Save HTML report\n"
    "        os.makedirs(output_dir, exist_ok=True)\n"
    "        timestamp = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n"
    "        report_path = f\"{output_dir}/drift_report_{timestamp}.html\"\n"
    "        report.save_html(report_path)\n"
    "        mlflow.log_artifact(report_path)\n"
    "        \n"
    "        # Extract results\n"
    "        results = report.as_dict()\n"
    "        drift_info = {\"timestamp\": timestamp}\n"
    "        for metric in results[\'metrics\']:\n"
    "            if metric[\'metric\'] == \'DatasetDriftMetric\':\n"
    "                drift_info[\'dataset_drift\'] = metric[\'result\'][\'dataset_drift\']\n"
    "                drift_info[\'drift_share\'] = metric[\'result\'][\'drift_share\']\n"
    "                drift_info[\'n_drifted\'] = metric[\'result\'][\'number_of_drifted_columns\']\n"
    "                drift_info[\'n_total\'] = metric[\'result\'][\'number_of_columns\']\n"
    "                \n"
    "                # Log to MLflow\n"
    "                mlflow.log_metric(\"dataset_drift_detected\", int(metric[\'result\'][\'dataset_drift\']))\n"
    "                mlflow.log_metric(\"drift_share\", metric[\'result\'][\'drift_share\'])\n"
    "                mlflow.log_metric(\"n_drifted_columns\", metric[\'result\'][\'number_of_drifted_columns\'])\n"
    "                mlflow.log_metric(\"total_columns\", metric[\'result\'][\'number_of_columns\'])\n"
    "        \n"
    "        # Model performance drift (if model provided)\n"
    "        if model_path and os.path.exists(model_path):\n"
    "            import joblib\n"
    "            model = joblib.load(model_path)\n"
    "            \n"
    "            exclude_cols = [\"target\", \"timestamp\", \"prediction\", \"prediction_proba\"]\n"
    "            feature_cols = [c for c in ref_df.columns if c not in exclude_cols]\n"
    "            \n"
    "            X_ref = ref_df[feature_cols]\n"
    "            y_ref = ref_df[\"target\"]\n"
    "            X_curr = curr_df[feature_cols]\n"
    "            y_curr = curr_df[\"target\"]\n"
    "            \n"
    "            ref_pred = model.predict(X_ref)\n"
    "            curr_pred = model.predict(X_curr)\n"
    "            \n"
    "            from sklearn.metrics import accuracy_score\n"
    "            ref_acc = accuracy_score(y_ref, ref_pred)\n"
    "            curr_acc = accuracy_score(y_curr, curr_pred)\n"
    "            \n"
    "            acc_drift = ref_acc - curr_acc\n"
    "            \n"
    "            mlflow.log_metric(\"ref_accuracy\", ref_acc)\n"
    "            mlflow.log_metric(\"curr_accuracy\", curr_acc)\n"
    "            mlflow.log_metric(\"accuracy_drift\", acc_drift)\n"
    "            drift_info[\'accuracy_drift\'] = acc_drift\n"
    "            \n"
    "            # Check accuracy drift threshold\n"
    "            if abs(acc_drift) > ACCURACY_DRIFT_THRESHOLD:\n"
    "                drift_info[\'performance_alert\'] = True\n"
    "                print(f\"⚠️ PERFORMANCE DRIFT: accuracy dropped by {acc_drift:.4f}\")\n"
    "        \n"
    "        # Alert if drift detected\n"
    "        alert_triggered = False\n"
    "        if drift_info.get(\'drift_share\', 0) > DRIFT_THRESHOLD:\n"
    "            drift_info[\'alert\'] = True\n"
    "            drift_info[\'alert_reason\'] = f\"Drift share {drift_info[\'drift_share\']:.1%} exceeds threshold {DRIFT_THRESHOLD}\"\n"
    "            alert_triggered = True\n"
    "        if drift_info.get(\'performance_alert\', False):\n"
    "            drift_info[\'alert\'] = True\n"
    "            drift_info[\'alert_reason\'] = drift_info.get(\'alert_reason\', \'\') + \"; \" + f\"Accuracy drift {abs(acc_drift):.4f} exceeds {ACCURACY_DRIFT_THRESHOLD}\"\n"
    "            alert_triggered = True\n"
    "        \n"
    "        if alert_triggered:\n"
    "            send_alert(drift_info, report_path)\n"
    "            mlflow.log_param(\"alert_triggered\", \"true\")\n"
    "        else:\n"
    "            mlflow.log_param(\"alert_triggered\", \"false\")\n"
    "        \n"
    "        # Save JSON summary\n"
    "        summary_path = f\"{output_dir}/drift_summary_{timestamp}.json\"\n"
    "        with open(summary_path, \'w\') as f:\n"
    "            json.dump(drift_info, f, indent=2)\n"
    "        mlflow.log_artifact(summary_path)\n"
    "        \n"
    "        return drift_info\n"
    "\n"
    "def send_alert(drift_info: dict, report_path: str):\n"
    "    \"\"\"Send email alert (configure SMTP).\"\"\"\n"
    "    if not ALERT_EMAIL:\n"
    "        print(\"ALERT_EMAIL not configured, skipping email\")\n"
    "        return\n"
    "    \n"
    "    subject = f\"⚠️ Data Drift Alert: {drift_info[\'drift_share\']:.1%} features drifted\"\n"
    "    body = f\"\n"
    "    Data drift detected in production data.\n"
    "\n"
    "    Drift share: {drift_info[\'drift_share\']:.1%}\n"
    "    Drifted columns: {drift_info[\'n_drifted\']} / {drift_info[\'n_total\']}\n"
    "    \n"
    "    Full report: {report_path}\n"
    "    \"\"\n"
    "    \n"
    "    # Implement SMTP sending here\n"
    "    print(f\"ALERT: {subject}\")\n"
    "\n"
    "if __name__ == \"__main__\":\n"
    "    import argparse\n"
    "    parser = argparse.ArgumentParser()\n"
    "    parser.add_argument(\"--reference\", required=True)\n"
    "    parser.add_argument(\"--current\", required=True)\n"
    "    parser.add_argument(\"--model\", default=None)\n"
    "    parser.add_argument(\"--output-dir\", default=\"./reports\")\n"
    "    args = parser.parse_args()\n"
    "    \n"
    "    result = check_drift(args.reference, args.current, args.model, args.output_dir)\n"
    "    print(json.dumps(result, indent=2))\n"
)

with open("monitoring.py", "w") as f:
    f.write(monitoring_script)

print("Enhanced monitoring script with MLflow created")
print("Run with: python monitoring.py --reference ref.csv --current curr.csv --model model.joblib")


## 7. Scheduled Monitoring (Cron/Airflow/Prefect)

In [ ]:
# Example cron entry (run daily at 2 AM)
# 0 2 * * * /path/to/venv/bin/python /path/to/monitoring.py \
#   --reference /data/reference.csv \
#   --current /data/current_$(date +\%Y\%m\%d).csv \
#   --model /models/production_model.joblib \
#   --output-dir /reports

# Airflow DAG example
airflow_dag = """
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'ml-engineer',
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    'drift_monitoring',
    default_args=default_args,
    schedule_interval='0 2 * * *',  # Daily at 2 AM
    start_date=datetime(2024, 1, 1),
    catchup=False,
) as dag:
    
    check_drift = BashOperator(
        task_id='check_drift',
        bash_command=(
            'cd /opt/ml/monitoring && '
            'python monitoring.py '
            '--reference /data/reference.csv '
            '--current /data/current_{{ ds }}.csv '
            '--model /models/production_model.joblib '
            '--output-dir /reports'
        ),
    )
    
    # Optional: retrain trigger
    retrain = BashOperator(
        task_id='retrain_if_needed',
        bash_command=(
            'cd /opt/ml && '
            'python retrain.py '
            '--trigger-file /reports/drift_trigger_{{ ds }}.flag'
        ),
    )
    
    check_drift >> retrain
"""

with open("drift_monitoring_dag.py", "w") as f:
    f.write(airflow_dag)

print("Airflow DAG example created: drift_monitoring_dag.py")

## 7. MLflow Dashboard Queries for Drift History

In [ ]:
# Query drift history from MLflow
client = MlflowClient()

# Get all drift monitoring runs
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.mlflow.runName LIKE 'drift_check_%'",
    order_by=["start_time DESC"],
    max_results=50
)

print(f"Total drift monitoring runs: {len(runs)}")

print(f"{'Run Name':<35} {'Date':<20} {'Drift Share':>12} {'Drifted':>10} {'Alert':>8}")
print('-' * 90)

for run in runs:
    name = run.data.tags.get('mlflow.runName', 'unnamed')
    date = datetime.fromtimestamp(run.info.start_time / 1000).strftime('%Y-%m-%d %H:%M')
    drift_share = run.data.metrics.get('drift_share', 0)
    n_drifted = run.data.metrics.get('n_drifted_columns', 0)
    alert = run.data.params.get('alert_triggered', run.data.params.get('performance_alert', 'false'))
    print(f"{name:<35} {date:<20} {drift_share:>11.2%} {n_drifted:>10} {alert:>8}")


## 8. Visualize Drift History

In [ ]:
# Plot drift history from MLflow
import matplotlib.pyplot as plt

drift_history = []
for run in runs:
    drift_share = run.data.metrics.get('drift_share', 0)
    n_drifted = run.data.metrics.get('n_drifted_columns', 0)
    accuracy_drift = run.data.metrics.get('accuracy_drift', 0)
    date = datetime.fromtimestamp(run.info.start_time / 1000)
    name = run.data.tags.get('mlflow.runName', '')
    
    drift_history.append({
        'date': date,
        'drift_share': drift_share,
        'n_drifted': n_drifted,
        'accuracy_drift': accuracy_drift,
        'name': name
    })

if drift_history:
    df_history = pd.DataFrame(drift_history).sort_values('date')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Drift share over time
    axes[0,0].plot(df_history['date'], df_history['drift_share'], 'o-', color='red')
    axes[0,0].axhline(y=0.3, color='red', linestyle='--', label='Alert threshold (30%)')
    axes[0,0].set_title('Drift Share Over Time')
    axes[0,0].set_ylabel('Drift Share')
    axes[0,0].legend()
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Number of drifted columns
    axes[0,1].bar(df_history['date'], df_history['n_drifted'], color='orange', alpha=0.7)
    axes[0,1].set_title('Number of Drifted Columns')
    axes[0,1].set_ylabel('Count')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Accuracy drift
    axes[1,0].plot(df_history['date'], df_history['accuracy_drift'], 'o-', color='blue')
    axes[1,0].axhline(y=0.05, color='red', linestyle='--', label='Alert threshold (5%)')
    axes[1,0].axhline(y=-0.05, color='red', linestyle='--')
    axes[1,0].set_title('Model Accuracy Drift')
    axes[1,0].set_ylabel('Accuracy Drift (ref - curr)')
    axes[1,0].legend()
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Drift share distribution
    axes[1,1].hist(df_history['drift_share'], bins=10, color='purple', alpha=0.7, edgecolor='black')
    axes[1,1].axvline(x=0.3, color='red', linestyle='--', label='Alert threshold')
    axes[1,1].set_title('Drift Share Distribution')
    axes[1,1].set_xlabel('Drift Share')
    axes[1,1].set_ylabel('Frequency')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No drift history available yet. Run drift checks to populate history.")

## Summary

| Drift Type | Evidently Preset | MLflow Integration | Alert Threshold |
|------------|------------------|-------------------|----------------|
| Data Drift | DataDriftPreset | Drift share, scores | Drift share > 30% |
| Data Quality | DataQualityPreset | HTML report | Missing values, outliers |
| Target Drift | TargetDriftPreset | HTML report | Target distribution shift |
| Prediction Drift | Custom | Accuracy/AUC drift | > 5% accuracy drop |

**Key Takeaways**:
1. **Monitor continuously**: Daily drift checks catch issues early
2. **Track everything in MLflow**: Drift metrics, reports, alerts all in one place
3. **Alert on both data and performance drift**: Data drift ≠ performance drift
4. **Automate with Airflow/Prefect**: Schedule daily checks, auto-retrain on alerts
5. **Visualize history**: MLflow dashboard shows drift trends over time

**Next Steps**:
- Add concept drift detection (relationship X→y changes)
- Implement statistical process control (SPC) charts
- Add feature importance drift tracking
- Build retraining trigger pipeline